# MIRAGE FlowPic Classification for Network Traffic Analysis
di Mario Gabriele Carofano

Questo notebook implementa una pipeline completa di **Traffic Classification** utilizzando la rappresentazione **FlowPic** sul dataset MIRAGE, dalla generazione del dataset fino alla valutazione finale dei modelli.

### Panoramica

Il workflow seguito è:

1. **Setup ambiente e riproducibilità**
	- Import librerie, costanti e funzioni custom.
	- Impostazione seed (`random`, `numpy`, `torch`) e scelta automatica del device (`mps` / `cuda` / `cpu`).

2. **Generazione del dataset (opzionale)**

	- Se `USE_PRECOMPUTED_DATASET` è impostato su `False`: il notebook genera il dataset FlowPic da zero utilizzando le funzioni custom implementate nel modulo `traffic_converter.py`.

	- Se `USE_PRECOMPUTED_DATASET` è impostato su `True`: il notebook carica da disco un dataset FlowPic già pre-calcolato.

3. **Preparazione dei dati**
	...

4. **Model selection**
	...

5. **Fase di training**
	...

6. **Fase di valutazione (su test set)**
	- Calcolo di **accuracy**, **confusion matrix** e **classification report**.

### Dataset (configurabile)

- **Formato**: `Tuple[np.ndarray, pd.DataFrame]`, dove:
	- `np.ndarray`: Dataset di istogrammi 2D delle sessioni di traffico, di shape `(N, 1, D, D)`, dove `N` è il numero di finestre temporali valide e `D` è la dimensione dell'istogramma, calcolata in base ai parametri `MTU` e `BIN_SIZE`.
	- `pd.DataFrame`: Metadati dei flussi validi, con le colonne `"FlowID"`, `"DatasetID"` e `"Label"`.
- **Shape dell'input**: `(N, 1, D, D)` — istogrammi 2D FlowPic delle sessioni di traffico.
- **Dimensione dell'istogramma (`D`)**: Calcolata in base alle costanti `MTU` e `BIN_SIZE`.

### Output
...

---

## Setup ambiente e riproducibilità

In [ ]:
#	LIBRARIES
#   ####################################################################    #

# Importing constant values
import importlib
import sys
sys.path.insert(1, '../src/')
import constants
importlib.reload(constants)

# Data loading and saving
from pathlib import Path
import pickle
import os

# Importing traffic converter functions
from traffic_converter import *

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Data preprocessing and evaluation
from preprocessing_functions import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Machine Learning and Quantum ML
from training_functions import *
from model_selection import *
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import pennylane as qml

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Other utilities
import copy
import datetime
import random
import time

In [ ]:
#	MACROS
#   ####################################################################    #

importlib.reload(constants)
from constants import RANDOM_SEED

#   ####################################################################    #

# 1. Configurazione del seed per la riproducibilità.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED) # Utile per hash di dizionari/set

# 2. Configurazione di PyTorch (CPU e GPU)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# 3. Configurazione di PyTorch per la riproducibilità su GPU (CUDNN)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 4. Configurazione del dispositivo (CPU o GPU)
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
# DEVICE = 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#	CLASSES
#	####################################################################    #

# TODO: implementare una classe Dataset personalizzata
# per il caricamento dei dati di traffico in formato FlowPic,
# e che gestisca le etichette in modo efficiente.

In [ ]:
#	FUNCTIONS
#   ####################################################################    #

def print_nonzero_elements(flow : np.ndarray) -> None:
	""" Stampa gli elementi non nulli di un array NumPy (FlowPic),
	mostrando le coordinate (riga, colonna) e il valore corrispondente.

	Args:
		flow (numpy.ndarray): Array 2D (FlowPic) da cui estrarre gli elementi non nulli.
	"""
    
	rows, cols = np.nonzero(flow)
	values = flow[rows, cols]

	print("Coordinate degli elementi non nulli:")
	for r, c, v in zip(rows, cols, values):
		print(f"[{r},{c}] = {v}", end=" | ")

	print()

	# end

def get_flowpic_dir(data_path: str) -> str:
	""" Sostituisce il nome 'flowpic' al posto di 'mirage' nel percorso del dataset. """

	p = Path(data_path)
	parts = p.parts

	if "mirage" not in parts:
		raise ValueError(
			f"Il percorso '{data_path}' non contiene la cartella 'mirage'.\n"
			"Assicurati di fornire un percorso valido che contenga 'mirage'."
		)

	idx = parts.index("mirage")
	new_parts = list(parts)
	new_parts[idx] = "flowpic"

	return str(Path(*new_parts))

	# end

---

In [ ]:
importlib.reload(constants)
from constants import (
	USE_PRECOMPUTED_DATASET,
    DATA_PATH, DATASET_NAME,
    MIN_TPS, MIN_PACKETS, MIN_DIM
)

#   ####################################################################    #

# Si impostano i filtri per la selezione dei flussi da elaborare.
filters = {
	'min_tps': MIN_TPS,
	# 'min_dim': MIN_DIM,
	# 'min_packets': MIN_PACKETS,
}

# Si impostano le variabili di debug per il caricamento del dataset.
debug = False
debug_cycle = False
flows_to_inspect = None

print("Loading dataset...")

flowpics_name = get_dataset_name(DATA_PATH, DATASET_NAME, debug)
flowpics_dir = get_flowpic_dir(DATA_PATH)
os.makedirs(flowpics_dir, exist_ok=True)

npy_path = os.path.join(flowpics_dir, f"{flowpics_name}.npy")
meta_path = os.path.join(flowpics_dir, f"{flowpics_name}_metadata.csv")

print(f"USE_PRECOMPUTED_DATASET impostato su {USE_PRECOMPUTED_DATASET}.\n")
if not USE_PRECOMPUTED_DATASET:

	histograms, metadata = mirage_pickle_converter(
		f"{DATA_PATH}/{DATASET_NAME}",
		filters, debug, debug_cycle, flows_to_inspect
	)

	if debug and flows_to_inspect is not None:
		
		for fid in flows_to_inspect:

			flow_metadata = metadata.loc[metadata['FlowID'] == fid]

			if flow_metadata.empty:
				print(f"[DEBUG] Flusso n.{fid} non trovato.")
				continue

			did = flow_metadata['DatasetID'].values[0]

			print(f"[DEBUG] Elaborazione del flusso n.{fid}")
			print(f"Shape: {histograms[did].shape}")
			print(f"Metadata:\n{flow_metadata.to_string(index=False)}")
			print_nonzero_elements(histograms[did][0])

			print(f"Min: {np.min(histograms[did])}")
			print(f"Max: {np.max(histograms[did])}")
			print(f"Mean: {np.mean(histograms[did])}")
			print(f"Std: {np.std(histograms[did])}")
			print(f"Sum: {np.sum(histograms[did])}\n")

			# end for fid
		# end if

	# Salvataggio degli istogrammi 2D FlowPic in formato NumPy.
	np.save(npy_path, histograms)
	print(f"Salvato dataset (shape={histograms.shape}) in: {npy_path}")

	# Salvataggio dei metadati in formato CSV.
	metadata.to_csv(meta_path, index=False)
	print(f"Salvati metadati (di {len(metadata)} righe) in: {meta_path} ")

elif USE_PRECOMPUTED_DATASET:
	if not os.path.exists(npy_path):
		raise FileNotFoundError(f"Array di istogrammi precomputati non trovato: {npy_path}")
	if not os.path.exists(meta_path):
		raise FileNotFoundError(f"DataFrame di metadati precomputati non trovato: {meta_path}")

	# Caricamento degli istogrammi 2D FlowPic precomputati.
	histograms = np.load(npy_path)
	print(f"Caricato dataset (shape={histograms.shape}) da: {npy_path}")

	# Caricamento dei metadati in formato CSV.
	metadata = pd.read_csv(meta_path)
	print(f"Caricati metadati (di {len(metadata)} righe) da: {meta_path} ")

# Per coerenza con l'altro notebook, si rinominano le variabili per il dataset e le etichette.
X_raw = histograms
y_raw = metadata["Label"].values

# Prima di procedere con split/training, si verifica che gli istogrammi e i metadati siano allineati.
assert histograms.shape[0] == len(metadata), "Disallineamento tra istogrammi e metadati!"
assert list(metadata["DatasetID"]) == list(range(len(metadata))), "DatasetID non contiguo!"